# Swin-T + UPerNet (ADE20K) — DIMER semantic-segmentation and bounded adaptation tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/swin-segmentation-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/swin-segmentation-pipeline/blob/main/tutorials/swin_segmentation_colab.ipynb) [![Python 3.10 required](https://img.shields.io/badge/Python-3.10%20required-3776ab?style=flat&logo=python&logoColor=white)](https://github.com/kurtvalcorza/swin-segmentation-pipeline/blob/main/README.md) [![Checkpoint](https://img.shields.io/badge/OpenMMLab-upernet__swin--t__ade20k__512x512__160k-ffcc4d?style=flat)](https://github.com/open-mmlab/mmsegmentation/tree/v1.2.2/configs/swin) [![Upstream](https://img.shields.io/badge/Upstream-microsoft%2FSwin--Transformer-181717?style=flat&logo=github&logoColor=white)](https://github.com/microsoft/Swin-Transformer) [![arXiv](https://img.shields.io/badge/arXiv-2103.14030-b31b1b.svg)](https://arxiv.org/abs/2103.14030) [![License](https://img.shields.io/badge/License-MIT-yellow.svg)](https://github.com/kurtvalcorza/swin-segmentation-pipeline/blob/main/LICENSE)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** pretrained ADE20K-150 semantic segmentation, and a bounded in-process adaptation workflow that re-heads Swin-T + UPerNet onto a custom segmentation vocabulary (background, road, structure), evaluates against a held-out split with mIoU and pixel accuracy, exports a portable adapter artifact, and reloads it with numerical verification

**This notebook is standalone.** It carries the repository's package (3 modules under `src/dimer_swin_segmentation/`, at revision `af738633e4ac`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the OpenMMLab checkpoint host (`download.openmmlab.com`) at the immutable MMSegmentation release-tag commit `c685fe6767c4cadf6b051983ca6208f1b9d1ccb8` (~240 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh CPython 3.10 runtime installs dependencies, stages and digest-verifies the pinned checkpoint, runs pretrained ADE20K inference on a demonstration scene, validates the 24-image segmentation adaptation dataset, splits it into train and validation sets, measures the pre-adaptation baseline, **runs the bounded fine-tune with frozen backbone**, re-evaluates on the held-out split, runs inference on an unseen test scene, exports the adapted artifact, reloads it from disk to verify numeric consistency, and writes machine-readable outputs with provenance. Nothing is skipped behind a default-off flag (NOTEBOOK_SPEC 2.0 §5, RUN7, FT2).

**Bring Your Own Data:** Two optional BYOD branches are included and both are disabled by default (`USE_BYOD_IMAGE = False`, `USE_BYOD_DATASET = False`). `USE_BYOD_IMAGE` runs your own image through the pretrained ADE20K model. `USE_BYOD_DATASET` takes your own labelled segmentation records and runs them through the full adaptation workflow under NOTEBOOK_SPEC 2.0 DAT14. Do not upload confidential or restricted imagery or data to hosted notebook environments.

Swin Transformer with Unified Perceptual Parsing (`upernet_swin_tiny_patch4_window7_512x512`) produces dense per-pixel semantic segmentations using hierarchical shifted windows and a multi-scale feature pyramid decoder. At inference, the pretrained model maps an RGB image to a 2-D class-index mask across the 150 ADE20K categories.

**The default path really adapts the model:** it generates a 24-scene deterministic segmentation dataset over a 3-class custom vocabulary (`background`, `road`, `structure`), validates the dataset contract (`core.dataset.vision.raster-mask`), partitions into train and validation splits, measures a pre-adaptation baseline on the held-out split, re-heads the decode and auxiliary heads, freezes the 28.3M Swin-T backbone, runs a bounded AdamW fine-tune loop, evaluates post-adaptation mIoU and pixel accuracy against the majority baseline, runs inference on an unseen test scene, exports `swin-segmentation-adapter-v1.pt`, and reloads the artifact from disk asserting exact numerical mask agreement.

**Trust boundary (MOD12).** The base checkpoint is a code-capable PyTorch `.pth` serialization. The carried `verify_snapshot` re-hashes it against the inline manifest and `MODEL_SPEC` before the pinned MMSegmentation loader deserializes it inside `mmengine`. The deserialization call is upstream's and is **not** a `weights_only` load; a matching digest proves byte identity with the pinned OpenMMLab distribution, not publisher authenticity.

**Learning objectives:** install the pinned Python 3.10 OpenMMLab runtime; read what the carried package guarantees; stage and digest-verify the immutable OpenMMLab checkpoint; run pretrained inference on an ADE20K demonstration scene; validate a multi-image segmentation dataset under `core.dataset.vision.raster-mask`; partition into train and validation splits; re-head the segmentation architecture onto a custom 3-class vocabulary; measure the pre-adaptation baseline; run bounded in-process fine-tuning with the Swin-T backbone frozen; evaluate post-adaptation mIoU, pixel accuracy, and per-class IoU against the majority baseline; run inference on an unseen test image; export the adapted artifact; and reload and numerically verify it.

**This notebook does not demonstrate:** real-world cityscapes deployment claims (the adaptation dataset is drawn in code, so the model learns these synthetic structures and nothing about street photographs); full network unfreezing without large annotated datasets (the default freezes the 28.3M Swin-T backbone); instance or panoptic segmentation; object detection; and depth estimation.

## Prerequisites

- **Runtime:** a **CPython 3.10** Jupyter kernel on Linux (the notebook asserts `sys.version_info[:2] == (3, 10)` and stops otherwise). The qualified OpenMMLab stack — torch 2.1.2 (CPU build), MMCV 2.1.0, MMEngine 0.10.7, MMSegmentation 1.2.2, NumPy 1.26.4 — has prebuilt wheels for Python 3.10 only. CPU is the default and only qualified path; no GPU is required.
- **Knowledge:** basic Python and PIL; dense semantic class masks; intersection-over-union (IoU) and pixel accuracy.
- **Data:** everything is generated deterministically in code by `samples.py`, requiring zero external dataset download: one 512×384 ADE20K demonstration scene and a 24-image custom segmentation dataset. Two optional BYOD branches are gated off by default.
- **External access:** the OpenMMLab checkpoint host (`download.openmmlab.com`) only, to fetch the pinned `open-mmlab/mmsegmentation:swin-tiny-patch4-window7-in1k-pre_upernet_8xb2-160k_ade20k-512x512` snapshot (~240 MB in total) at revision `c685fe6767c4…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's tools/pins.txt at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `numpy`, `PIL` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    '--extra-index-url',
    'https://download.pytorch.org/whl/cpu',
    '--find-links',
    'https://download.openmmlab.com/mmcv/dist/cpu/torch2.1/index.html',
    'torch==2.1.2+cpu',
    'torchvision==0.16.2+cpu',
    'mmengine==0.10.7',
    'mmcv==2.1.0',
    'mmsegmentation==1.2.2',
    'ftfy==6.3.1',
    'regex==2024.11.6',
    'numpy==1.26.4',
    'opencv-python==4.10.0.84',
    'pillow==11.3.0',
]
NOTEBOOK_SOURCE = {
    'repository': 'swin-segmentation-pipeline',
    'repository_revision': 'af738633e4acc3b374a10d78f756c53b1bb09100',
    'embedded_module': 'src/dimer_swin_segmentation/runtime.py',
    'embedded_modules': ['src/dimer_swin_segmentation/metrics.py', 'src/dimer_swin_segmentation/runtime.py', 'src/dimer_swin_segmentation/samples.py'],
    'module_sha256': 'ebf40ced226ff9a6fb3337eec18b8963acbe38924664f4be89ac8b74c782c10a',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, numpy, PIL
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'numpy': numpy.__version__, 'PIL': PIL.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/dimer_swin_segmentation/` @ `af738633e4ac`)

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/dimer_swin_segmentation/metrics.py`

In [ ]:
"""Semantic-segmentation metric helpers carried by the standalone tutorial (NOTEBOOK_SPEC 1.1 EVAL2).

`semantic_iou` is the repository's segmentation metric: per-class intersection/union aggregated over
every (prediction, reference) pair, mean IoU over the classes that occur in the sample (union > 0),
and pixel accuracy over the labelled pixels. `majority_class_baseline` scores the constant predictor
that paints every pixel with the sample's most frequent reference class — a descriptive reference
derived from the same tiny sample, not an independent benchmark. `ade20k_raw_to_indices` maps a raw
ADE20K annotation (`0` = unlabelled, `1..150` = classes) onto the model's class indices `0..149` with
`0` sent to the ignore index, the same `reduce_zero_label=True` convention the pinned MMSegmentation
config uses. Pure NumPy; no model logic lives here.
"""

from __future__ import annotations

from collections.abc import Sequence
from typing import Any

import numpy as np

ADE20K_NUM_CLASSES = 150
IGNORE_INDEX = 255


def ade20k_raw_to_indices(raw: np.ndarray, *, num_classes: int = ADE20K_NUM_CLASSES) -> np.ndarray:
    """Raw ADE20K labels (0 = ignore, 1..num_classes) -> model indices 0..num_classes-1, ignore -> 255."""
    array = np.asarray(raw)
    if array.ndim != 2:
        raise ValueError(f"expected a 2-D label map, got shape {array.shape}")
    if array.min() < 0 or array.max() > num_classes:
        observed = f"{int(array.min())}..{int(array.max())}"
        raise ValueError(f"raw ADE20K labels must lie in 0..{num_classes}: {observed}")
    out = np.full(array.shape, IGNORE_INDEX, dtype=np.uint16)
    labelled = array > 0
    out[labelled] = array[labelled].astype(np.uint16) - 1
    return out


def _pairs(
    predictions: Sequence[np.ndarray], references: Sequence[np.ndarray], ignore_index: int
) -> list[tuple[np.ndarray, np.ndarray, np.ndarray]]:
    if len(predictions) != len(references):
        raise ValueError(f"{len(predictions)} predictions but {len(references)} references")
    if not predictions:
        raise ValueError("at least one prediction/reference pair is required")
    out = []
    for index, (prediction, reference) in enumerate(zip(predictions, references, strict=True)):
        pred = np.asarray(prediction)
        ref = np.asarray(reference)
        if pred.ndim != 2 or pred.shape != ref.shape:
            raise ValueError(
                f"pair {index}: prediction {pred.shape} and reference {ref.shape} must be equal 2-D shapes"
            )
        out.append((pred.astype(np.int64), ref.astype(np.int64), ref != ignore_index))
    return out


def semantic_iou(
    predictions: Sequence[np.ndarray],
    references: Sequence[np.ndarray],
    *,
    num_classes: int = ADE20K_NUM_CLASSES,
    ignore_index: int = IGNORE_INDEX,
) -> dict[str, Any]:
    """Aggregate per-class IoU, mean IoU over classes present (union > 0) and pixel accuracy.

    `predictions` and `references` are equal-length sequences of equal-shape 2-D class-index maps
    (model indices `0..num_classes-1`); reference pixels equal to `ignore_index` are excluded.
    """
    intersections = np.zeros(num_classes, dtype=np.int64)
    unions = np.zeros(num_classes, dtype=np.int64)
    correct = 0
    valid_pixels = 0
    for pred, ref, valid in _pairs(predictions, references, ignore_index):
        if valid.any() and (pred[valid].min() < 0 or pred[valid].max() >= num_classes):
            raise ValueError(f"prediction indices must lie in 0..{num_classes - 1}")
        if valid.any() and ref[valid].max() >= num_classes:
            raise ValueError(f"reference indices must lie in 0..{num_classes - 1} or equal ignore_index")
        correct += int(((pred == ref) & valid).sum())
        valid_pixels += int(valid.sum())
        pred_hist = np.bincount(pred[valid], minlength=num_classes)[:num_classes]
        ref_hist = np.bincount(ref[valid], minlength=num_classes)[:num_classes]
        both = np.bincount(pred[valid & (pred == ref)], minlength=num_classes)[:num_classes]
        intersections += both
        unions += pred_hist + ref_hist - both
    present = np.flatnonzero(unions > 0)
    per_class = [
        {
            "class_id": int(class_id),
            "intersection_pixels": int(intersections[class_id]),
            "union_pixels": int(unions[class_id]),
            "iou": float(intersections[class_id] / unions[class_id]),
        }
        for class_id in present
    ]
    return {
        "miou": float(np.mean([row["iou"] for row in per_class])) if per_class else float("nan"),
        "pixel_accuracy": float(correct / valid_pixels) if valid_pixels else float("nan"),
        "valid_pixels": valid_pixels,
        "classes_with_union": len(per_class),
        "n_images": len(predictions),
        "per_class": per_class,
    }


def majority_class_baseline(
    references: Sequence[np.ndarray],
    *,
    num_classes: int = ADE20K_NUM_CLASSES,
    ignore_index: int = IGNORE_INDEX,
) -> dict[str, Any]:
    """Score the constant predictor painting every pixel with the sample's most frequent reference class."""
    refs = [np.asarray(reference).astype(np.int64) for reference in references]
    if not refs:
        raise ValueError("at least one reference is required")
    counts = np.zeros(num_classes, dtype=np.int64)
    for ref in refs:
        valid = ref != ignore_index
        counts += np.bincount(ref[valid], minlength=num_classes)[:num_classes]
    if not counts.any():
        raise ValueError("references contain no labelled pixels")
    majority = int(counts.argmax())
    constant = [np.full(ref.shape, majority, dtype=np.int64) for ref in refs]
    scored = semantic_iou(constant, refs, num_classes=num_classes, ignore_index=ignore_index)
    return {"majority_class_id": majority, "miou": scored["miou"], "pixel_accuracy": scored["pixel_accuracy"]}

**Module 2/3:** `src/dimer_swin_segmentation/runtime.py` (carried verbatim; see the note above)

In [ ]:
from __future__ import annotations

import hashlib
import importlib.metadata
import json
import os
import tempfile
import urllib.request
from collections.abc import Callable, Iterable, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image

# standalone rewrite (build_notebook.py): `from .metrics import majority_class_baseline, semantic_iou` removed — names are kernel globals defined by the carried modules

MODEL_SPEC = {
    "runtime_id": "swin-t-upernet-ade20k-mmseg-v1.2.2",
    "architecture": "Swin-T + UPerNet",
    "task": "semantic-segmentation",
    "dataset": "ADE20K",
    "mmsegmentation_version": "1.2.2",
    "mmcv_version": "2.1.0",
    "mmengine_version": "0.10.7",
    "torch_version": "2.1.2",
    "config": "swin/swin-tiny-patch4-window7-in1k-pre_upernet_8xb2-160k_ade20k-512x512.py",
    "config_source_revision": "open-mmlab/mmsegmentation@v1.2.2",
    "checkpoint_url": "https://download.openmmlab.com/mmsegmentation/v0.5/swin/upernet_swin_tiny_patch4_window7_512x512_160k_ade20k_pretrain_224x224_1K/upernet_swin_tiny_patch4_window7_512x512_160k_ade20k_pretrain_224x224_1K_20210531_112542-e380ad3e.pth",
    "checkpoint_size_bytes": 240154742,
    "checkpoint_sha256": "e380ad3e5d94060d89e4b62b5d393cdcc7f1f3406b1d46bcab547d3c276b6064",
    "upstream_reported_miou": 44.41,
    "num_classes": 150,
}

ARTIFACT_FORMAT = "dimer_swin_segmentation_adapter_v1"
DEFAULT_ADAPT_EPOCHS = 3
DEFAULT_ADAPT_BATCH_SIZE = 4
DEFAULT_ADAPT_LEARNING_RATE = 1e-4
DEFAULT_ADAPT_SEED = 20260915


@dataclass(frozen=True)
class SegmentationResult:
    image_id: str
    mask: np.ndarray
    classes_present: tuple[int, ...]

    def summary(self) -> dict:
        return {
            "image_id": self.image_id,
            "shape": list(self.mask.shape),
            "classes_present": list(self.classes_present),
            "num_classes_present": len(self.classes_present),
        }

    def save_mask(self, path: str | os.PathLike) -> Path:
        target = Path(path)
        target.parent.mkdir(parents=True, exist_ok=True)
        Image.fromarray(self.mask.astype(np.uint8), mode="L").save(target)
        return target


def _sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def _version(dist: str) -> str:
    return importlib.metadata.version(dist)


# ---------------------------------------------------------------------------------------------
# Fleet snapshot scheme (DIMER standalone carrier). The identity constants below name the OpenMMLab
# distribution: MODEL_ID is the config recipe inside the pinned package, MODEL_REVISION the upstream
# git commit of that package's release tag (the config source), and the manifest pins the checkpoint
# bytes. The checkpoint host is download.openmmlab.com, not the Hugging Face Hub, so the staging
# downloader is the pinned URL in MODEL_SPEC rather than hf_hub_download.
# ---------------------------------------------------------------------------------------------
MODEL_ID = "open-mmlab/mmsegmentation:swin-tiny-patch4-window7-in1k-pre_upernet_8xb2-160k_ade20k-512x512"
MODEL_REVISION = "c685fe6767c4cadf6b051983ca6208f1b9d1ccb8"
MODEL_LICENSE = "Apache-2.0"
MODEL_KEY = "swin-t-upernet-ade20k"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"
WEIGHTS_FILE = "upernet_swin_tiny_patch4_window7_512x512_160k_ade20k_pretrain_224x224_1K_20210531_112542-e380ad3e.pth"
MAX_PIXELS = 64_000_000  # validate_image ceiling


def verify_snapshot(path: str | os.PathLike | None = None) -> dict:
    """Check a local snapshot against its manifest; raise naming the first mismatch."""
    root = Path(path or DEFAULT_WEIGHTS_DIR)
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    entries = {entry["path"]: entry for entry in manifest.get("files", [])}
    pinned = entries.get(WEIGHTS_FILE)
    if pinned is None or pinned["sha256"] != MODEL_SPEC["checkpoint_sha256"] or pinned["bytes"] != MODEL_SPEC["checkpoint_size_bytes"]:
        raise ValueError(f"manifest entry for {WEIGHTS_FILE} does not match the checkpoint digest pinned in MODEL_SPEC")
    for entry in manifest.get("files", []):
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {"path": str(root), **manifest}


def _openmmlab_download(relative_path: str, root: Path) -> None:
    """Fetch the pinned OpenMMLab checkpoint into the snapshot directory (the only manifest entry)."""
    if relative_path != WEIGHTS_FILE:
        raise ValueError(f"no pinned download source for {relative_path}")
    target = root / relative_path
    target.parent.mkdir(parents=True, exist_ok=True)
    fd, temporary_name = tempfile.mkstemp(prefix="dimer-swin-", suffix=".pth", dir=root)
    os.close(fd)
    temporary = Path(temporary_name)
    try:
        with urllib.request.urlopen(MODEL_SPEC["checkpoint_url"], timeout=120) as response, temporary.open("wb") as out:
            while True:
                block = response.read(1024 * 1024)
                if not block:
                    break
                out.write(block)
        temporary.replace(target)
    finally:
        if temporary.exists():
            temporary.unlink()


def stage_missing_files(
    path: str | os.PathLike | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the checkpoint). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them from {MODEL_SPEC['checkpoint_url']}"
        )
    fetch = downloader or _openmmlab_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def verify_runtime_versions() -> dict[str, str]:
    expected = {
        "mmsegmentation": MODEL_SPEC["mmsegmentation_version"],
        "mmcv": MODEL_SPEC["mmcv_version"],
        "mmengine": MODEL_SPEC["mmengine_version"],
    }
    actual = {name: _version(name) for name in expected}
    for name, expected_version in expected.items():
        if actual[name] != expected_version:
            raise RuntimeError(
                f"Unsupported {name} {actual[name]}; this runtime is qualified for {expected_version}."
            )
    return actual


def resolve_packaged_config() -> Path:
    import mmseg

    config = (
        Path(mmseg.__file__).resolve().parent
        / ".mim"
        / "configs"
        / MODEL_SPEC["config"]
    )
    if not config.is_file():
        raise RuntimeError(
            f"The pinned MMSegmentation package does not contain the expected config: {config}"
        )
    return config


def acquire_verified_checkpoint(cache_dir: str | os.PathLike = ".dimer-models") -> Path:
    root = Path(cache_dir).expanduser().resolve()
    root.mkdir(parents=True, exist_ok=True)
    target = root / Path(MODEL_SPEC["checkpoint_url"]).name

    def valid(path: Path) -> bool:
        return (
            path.is_file()
            and path.stat().st_size == MODEL_SPEC["checkpoint_size_bytes"]
            and _sha256(path) == MODEL_SPEC["checkpoint_sha256"]
        )

    if valid(target):
        return target
    if target.exists():
        target.unlink()

    fd, temporary_name = tempfile.mkstemp(prefix="dimer-swin-segmentation-", suffix=".pth", dir=root)
    os.close(fd)
    temporary = Path(temporary_name)
    try:
        with urllib.request.urlopen(MODEL_SPEC["checkpoint_url"], timeout=120) as response, temporary.open("wb") as out:
            while True:
                block = response.read(1024 * 1024)
                if not block:
                    break
                out.write(block)
        if temporary.stat().st_size != MODEL_SPEC["checkpoint_size_bytes"]:
            raise RuntimeError(
                f"Checkpoint size mismatch: got {temporary.stat().st_size}, expected {MODEL_SPEC['checkpoint_size_bytes']}."
            )
        digest = _sha256(temporary)
        if digest != MODEL_SPEC["checkpoint_sha256"]:
            raise RuntimeError(
                f"Checkpoint SHA-256 mismatch: got {digest}, expected {MODEL_SPEC['checkpoint_sha256']}."
            )
        temporary.replace(target)
    finally:
        if temporary.exists():
            temporary.unlink()
    return target


def validate_image(path: str | os.PathLike, *, max_pixels: int = 64_000_000) -> dict:
    image_path = Path(path).expanduser().resolve()
    if not image_path.is_file():
        raise ValueError(f"Image does not exist: {image_path}")
    try:
        with Image.open(image_path) as image:
            image.verify()
        with Image.open(image_path) as image:
            width, height = image.size
            mode = image.mode
    except Exception as exc:
        raise ValueError(f"Input is not a readable image: {image_path}: {exc}") from exc
    if width < 1 or height < 1:
        raise ValueError(f"Image dimensions must be positive; got {width}x{height}.")
    if width * height > max_pixels:
        raise ValueError(
            f"Image has {width * height:,} pixels; ceiling is {max_pixels:,}. Resize before inference."
        )
    return {"path": str(image_path), "width": width, "height": height, "mode": mode}


INPUT_SCHEMA: dict = {
    "input": "one or more image files readable by Pillow (any mode), given by path",
    "pixels": [1, MAX_PIXELS],
    "classes": "150 ADE20K categories in MMSegmentation order (or custom classes when adapted)",
    "preprocessing": "MMSegmentation test pipeline of the pinned config (resize to 512-scale, normalise); nothing is altered by this module",
}


def validate_inputs(
    images: str | os.PathLike | Iterable[str | os.PathLike],
    *,
    names: Iterable[str] | None = None,
) -> dict:
    """Validation stage: return the input manifest (schema, per-image observations, verdict)."""
    paths = [images] if isinstance(images, (str, os.PathLike)) else list(images)
    observed = [validate_image(p) for p in paths]
    ids = list(names) if names is not None else [Path(o["path"]).name for o in observed]
    if len(ids) != len(observed):
        raise ValueError("names must have one entry per image")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [{"id": ids[i], **o} for i, o in enumerate(observed)],
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    results: Iterable[SegmentationResult | dict],
    ground_truth: Iterable[np.ndarray] | None = None,
    *,
    sample_kind: str = "synthetic",
    num_classes: int | None = None,
) -> dict:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``ground_truth`` (one 2-D map of class indices ``0..num_classes-1`` per result, ``255`` = ignore)
    the report carries ``semantic_iou`` (mean IoU over classes present, pixel accuracy, per-class IoU)
    and the ``majority_class_baseline`` with verdict ``sample-sanity``. Without ground truth the verdict
    is ``not-measurable``.
    """
    items = list(results)
    rows = [r.summary() if isinstance(r, SegmentationResult) else dict(r) for r in items]
    eval_num_classes = num_classes if num_classes is not None else MODEL_SPEC["num_classes"]
    base = {
        "task": f"semantic-segmentation ({eval_num_classes} classes)",
        "score_semantics": "argmax class per pixel; no per-pixel confidence is exposed",
        "sample_kind": sample_kind,
        "n_images": len(rows),
        "num_classes": eval_num_classes,
        "context": {
            "upstream_reported_miou": MODEL_SPEC["upstream_reported_miou"],
            "note": "upstream full-ADE20K mIoU as reported by OpenMMLab; not measured here",
        },
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if ground_truth is None:
        return {
            **base,
            "metrics": [],
            "baselines": [],
            "verdict": "not-measurable",
            "reason": "no ground-truth masks were supplied for the evaluated images",
            "needs": (
                "Ground-truth indexed masks (indices 0..num_classes-1, 255 = ignore) scored with "
                "semantic_iou (mean IoU over classes present, pixel accuracy, per-class IoU) against "
                "the majority_class_baseline; a labelled set from the deployment domain for any generalisable claim."
            ),
        }
    if not all(isinstance(item, SegmentationResult) for item in items):
        raise TypeError("ground_truth evaluation needs SegmentationResult inputs (the predicted masks)")
    references = [np.asarray(reference) for reference in ground_truth]
    scored = semantic_iou([item.mask for item in items], references, num_classes=eval_num_classes)
    baseline = majority_class_baseline(references, num_classes=eval_num_classes)
    estimation = (
        f"sample of {scored['n_images']} image(s) / {scored['valid_pixels']} labelled pixels, "
        "aggregate IoU, no dispersion estimate"
    )
    return {
        **base,
        "metrics": [
            {"id": "semantic_iou", "metric": "miou", "value": scored["miou"], "estimation": estimation},
            {"id": "semantic_iou", "metric": "pixel_accuracy", "value": scored["pixel_accuracy"], "estimation": estimation},
        ],
        "per_class": scored["per_class"],
        "classes_with_union": scored["classes_with_union"],
        "baselines": [
            {
                "id": "majority_class_baseline",
                "majority_class_id": baseline["majority_class_id"],
                "miou": baseline["miou"],
                "pixel_accuracy": baseline["pixel_accuracy"],
                "note": "constant predictor of the sample's most frequent class; derived from the same sample",
            }
        ],
        "verdict": "sample-sanity",
        "reason": f"{scored['n_images']} labelled image(s); evaluated locally",
        "needs": "a representative labelled holdout from the deployment domain for any generalisable mIoU claim",
    }


def rehead_model(model: Any, class_names: Sequence[str], *, seed: int = DEFAULT_ADAPT_SEED) -> tuple[int, int]:
    """Re-head the UPerNet decode head and auxiliary head onto a custom class vocabulary."""
    num_classes = len(class_names)
    if num_classes < 1:
        raise ValueError("class_names must contain at least 1 class")

    if hasattr(model, "dataset_meta") and isinstance(model.dataset_meta, dict):
        model.dataset_meta["classes"] = tuple(class_names)

    in_decode = 512
    in_aux = 256

    decode_head = getattr(model, "decode_head", None)
    if decode_head is not None and hasattr(decode_head, "conv_seg"):
        import torch
        import torch.nn as nn

        torch.manual_seed(seed)
        in_decode = decode_head.conv_seg.in_channels
        new_conv = nn.Conv2d(in_decode, num_classes, kernel_size=1)
        nn.init.kaiming_normal_(new_conv.weight, mode="fan_out", nonlinearity="relu")
        if new_conv.bias is not None:
            nn.init.constant_(new_conv.bias, 0.0)
        decode_head.conv_seg = new_conv
        decode_head.num_classes = num_classes
        decode_head.out_channels = num_classes

    aux_head = getattr(model, "auxiliary_head", None)
    if aux_head is not None and hasattr(aux_head, "conv_seg"):
        import torch
        import torch.nn as nn

        torch.manual_seed(seed)
        in_aux = aux_head.conv_seg.in_channels
        new_conv_aux = nn.Conv2d(in_aux, num_classes, kernel_size=1)
        nn.init.kaiming_normal_(new_conv_aux.weight, mode="fan_out", nonlinearity="relu")
        if new_conv_aux.bias is not None:
            nn.init.constant_(new_conv_aux.bias, 0.0)
        aux_head.conv_seg = new_conv_aux
        aux_head.num_classes = num_classes
        aux_head.out_channels = num_classes

    return in_decode, in_aux


def freeze_backbone(model: Any) -> int:
    """Freeze all parameters in the model backbone; return total frozen parameters."""
    frozen = 0
    backbone = getattr(model, "backbone", None)
    if backbone is not None and hasattr(backbone, "parameters"):
        for p in backbone.parameters():
            p.requires_grad = False
            frozen += p.numel()
    return frozen


class DimerSwinSegmenter:
    """Public semantic-segmentation task-inference and E2E adaptation API for Swin-T UPerNet."""

    def __init__(
        self,
        *,
        cache_dir: str | os.PathLike = ".dimer-models",
        device: str = "cpu",
        checkpoint: str | os.PathLike | None = None,
        source: str = "openmmlab-cache",
        class_names: Sequence[str] | None = None,
        freeze_backbone_weights: bool = False,
        seed: int = DEFAULT_ADAPT_SEED,
    ):
        versions = verify_runtime_versions()
        checkpoint = Path(checkpoint) if checkpoint is not None else acquire_verified_checkpoint(cache_dir)
        config = resolve_packaged_config()
        from mmseg.apis import init_model

        self.model = init_model(str(config), str(checkpoint), device=device)
        self.device = device
        self.checkpoint = checkpoint
        self.config = config
        self.versions = versions
        self.source = source
        self.adapted = False
        self.seed = seed
        self.base_state_digest = _sha256(self.checkpoint) if self.checkpoint.is_file() else ""

        if class_names is not None:
            rehead_model(self.model, class_names, seed=seed)
            if freeze_backbone_weights:
                freeze_backbone(self.model)
            self.classes = tuple(class_names)
        else:
            self.classes = tuple(self.model.dataset_meta.get("classes", ()))
            if len(self.classes) != MODEL_SPEC["num_classes"]:
                raise RuntimeError(
                    f"Expected {MODEL_SPEC['num_classes']} ADE20K classes, got {len(self.classes)}."
                )

    @classmethod
    def from_pretrained(
        cls,
        *,
        device: str = "cpu",
        weights_dir: str | os.PathLike | None = None,
        allow_download: bool = False,
        class_names: Sequence[str] | None = None,
        freeze_backbone: bool = False,
        seed: int = DEFAULT_ADAPT_SEED,
    ) -> DimerSwinSegmenter:
        """Load from the fleet snapshot directory, verify digest, and optionally re-head for custom classes."""
        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        if not (root / MANIFEST_NAME).is_file():
            raise FileNotFoundError(
                f"no snapshot manifest at {root}; use DimerSwinSegmenter(cache_dir=...) for the cache path"
            )
        stage_missing_files(root, allow_download=allow_download)
        verify_snapshot(root)
        return cls(
            device=device,
            checkpoint=root / WEIGHTS_FILE,
            source="local-snapshot",
            class_names=class_names,
            freeze_backbone_weights=freeze_backbone,
            seed=seed,
        )

    def predict(self, image: str | os.PathLike | Image.Image | np.ndarray) -> SegmentationResult:
        if isinstance(image, Image.Image):
            info = {"path": "in_memory_image.png", "width": image.width, "height": image.height, "mode": image.mode}
            img_input = np.array(image.convert("RGB"))
        elif isinstance(image, np.ndarray):
            info = {"path": "in_memory_array.png", "width": image.shape[1], "height": image.shape[0], "mode": "RGB"}
            img_input = image
        else:
            info = validate_image(image)
            img_input = info["path"]

        from mmseg.apis import inference_model

        sample = inference_model(self.model, img_input)
        mask = sample.pred_sem_seg.data.squeeze(0).detach().cpu().numpy().astype(np.uint8, copy=False)
        if mask.ndim != 2:
            raise RuntimeError(f"Expected a 2-D semantic mask; got shape {mask.shape}.")
        classes_present = tuple(int(x) for x in np.unique(mask))
        return SegmentationResult(
            image_id=Path(info["path"]).name,
            mask=mask,
            classes_present=classes_present,
        )

    def finetune(
        self,
        records: Sequence[dict[str, Any]],
        *,
        epochs: int = DEFAULT_ADAPT_EPOCHS,
        batch_size: int = DEFAULT_ADAPT_BATCH_SIZE,
        learning_rate: float = DEFAULT_ADAPT_LEARNING_RATE,
        seed: int = DEFAULT_ADAPT_SEED,
        freeze_backbone_weights: bool = True,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded in-process fine-tuning on ``records`` using MMSegmentation loss and AdamW."""
        import torch

        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        validate_dataset(records, self.classes, epochs=epochs)
        if not isinstance(batch_size, int) or isinstance(batch_size, bool) or batch_size < 1:
            raise ValueError(f"batch_size must be a positive int, got {batch_size!r}")
        if not isinstance(learning_rate, (int, float)) or isinstance(learning_rate, bool):
            raise ValueError(f"learning_rate must be a number, got {learning_rate!r}")
        if not 0.0 < float(learning_rate) <= 1.0:
            raise ValueError(f"learning_rate must be in (0, 1], got {learning_rate!r}")

        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        rng = np.random.default_rng(seed)

        if freeze_backbone_weights:
            freeze_backbone(self.model)

        trainable = [p for p in self.model.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(trainable, lr=learning_rate, weight_decay=1e-4)

        from mmengine.structures import PixelData
        from mmseg.structures import SegDataSample

        n_records = len(records)
        epoch_losses: list[float] = []

        for epoch in range(epochs):
            self.model.train()
            order = rng.permutation(n_records)
            running_loss = 0.0
            n_batches = 0

            for start in range(0, n_records, batch_size):
                batch_indices = order[start : start + batch_size]
                batch_records = [records[i] for i in batch_indices]

                img_tensors = []
                data_samples = []
                for rec in batch_records:
                    img_raw = rec["image"]
                    if isinstance(img_raw, (str, Path)):
                        with Image.open(img_raw) as opened:
                            arr = np.array(opened.convert("RGB"))
                    elif isinstance(img_raw, Image.Image):
                        arr = np.array(img_raw.convert("RGB"))
                    else:
                        arr = np.asarray(img_raw)

                    mask_raw = rec["mask"]
                    if isinstance(mask_raw, (str, Path)):
                        with Image.open(mask_raw) as opened_m:
                            mask_arr = np.array(opened_m)
                    elif isinstance(mask_raw, Image.Image):
                        mask_arr = np.array(mask_raw)
                    else:
                        mask_arr = np.asarray(mask_raw)

                    h, w = arr.shape[:2]
                    img_t = torch.from_numpy(arr).permute(2, 0, 1).to(self.device).float()
                    mask_t = torch.from_numpy(mask_arr).long().unsqueeze(0).to(self.device)

                    sample = SegDataSample()
                    sample.gt_sem_seg = PixelData(data=mask_t)
                    sample.set_metainfo({"img_shape": (h, w), "ori_shape": (h, w), "pad_shape": (h, w)})

                    img_tensors.append(img_t)
                    data_samples.append(sample)

                batch_data = self.model.data_preprocessor(
                    {"inputs": img_tensors, "data_samples": data_samples}, training=True
                )
                losses = self.model(**batch_data, mode="loss")
                parsed_loss, _ = self.model.parse_losses(losses)

                optimizer.zero_grad(set_to_none=True)
                parsed_loss.backward()
                optimizer.step()

                running_loss += float(parsed_loss.detach().cpu())
                n_batches += 1

            mean_epoch_loss = running_loss / max(1, n_batches)
            epoch_losses.append(mean_epoch_loss)
            if progress is not None:
                progress({"epoch": epoch + 1, "epochs": epochs, "loss": mean_epoch_loss})

        self.model.eval()
        self.adapted = True

        total_params = sum(p.numel() for p in self.model.parameters()) if hasattr(self.model, "parameters") else 0
        trainable_params = sum(p.numel() for p in trainable)

        return {
            "epochs": epochs,
            "batch_size": batch_size,
            "learning_rate": float(learning_rate),
            "seed": seed,
            "freeze_backbone_weights": freeze_backbone_weights,
            "trainable_parameters": trainable_params,
            "total_parameters": total_params,
            "epoch_losses": epoch_losses,
            "final_loss": epoch_losses[-1] if epoch_losses else None,
            "device": self.device,
            "classes": list(self.classes),
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    def evaluate(
        self,
        records: Sequence[dict[str, Any]],
        *,
        sample_kind: str = "validation",
    ) -> dict[str, Any]:
        """Evaluate the model on a sequence of labelled dataset records."""
        results = [self.predict(rec["image"]) for rec in records]
        references = [np.asarray(rec["mask"]) for rec in records]
        return evaluation_report(
            results,
            references,
            sample_kind=sample_kind,
            num_classes=len(self.classes),
        )

    def save_artifact(self, path: str | os.PathLike, *, notes: str | None = None) -> dict[str, Any]:
        """Write adapted weights, vocabulary, and provenance as a portable .pt artifact."""
        import torch

        target = Path(path)
        target.parent.mkdir(parents=True, exist_ok=True)
        payload = {
            "format": ARTIFACT_FORMAT,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
            "model_key": MODEL_KEY,
            "checkpoint_sha256": MODEL_SPEC["checkpoint_sha256"],
            "class_names": list(self.classes),
            "adapted": self.adapted,
            "base_state_digest": self.base_state_digest,
            "notes": notes or "",
            "state_dict": {k: v.detach().cpu() for k, v in self.model.state_dict().items()},
        }
        torch.save(payload, target)
        return {
            "path": str(target),
            "bytes": target.stat().st_size,
            "sha256": _sha256(target),
            "format": ARTIFACT_FORMAT,
            "class_names": list(self.classes),
            "tensors": len(payload["state_dict"]),
            "base_state_digest": self.base_state_digest,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    @classmethod
    def load_artifact(
        cls,
        path: str | os.PathLike,
        *,
        weights_dir: str | os.PathLike | None = None,
        device: str = "cpu",
    ) -> DimerSwinSegmenter:
        """Rebuild an adapted segmentation pipeline from an exported .pt artifact."""
        import torch

        target = Path(path)
        if not target.is_file():
            raise FileNotFoundError(f"Artifact not found: {target}")
        payload = torch.load(target, map_location="cpu", weights_only=True)
        if payload.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {payload.get('format')!r} != {ARTIFACT_FORMAT!r}")
        if payload.get("model_id") != MODEL_ID or payload.get("model_revision") != MODEL_REVISION:
            raise ValueError(
                f"artifact built on {payload.get('model_id')}@{payload.get('model_revision')}, "
                f"package pins {MODEL_ID}@{MODEL_REVISION}"
            )
        if payload.get("model_key") != MODEL_KEY:
            raise ValueError(f"artifact model_key {payload.get('model_key')!r} != {MODEL_KEY!r}")

        names = tuple(payload["class_names"])
        pipe = cls.from_pretrained(
            device=device,
            weights_dir=weights_dir,
            class_names=names,
        )
        pipe.model.load_state_dict(payload["state_dict"], strict=True)
        pipe.adapted = True
        pipe.source = f"artifact:{target.name}"
        return pipe

    def provenance(self) -> dict:
        import platform

        import torch

        return {
            "runtime": MODEL_SPEC,
            "effective": {
                "python": platform.python_version(),
                "torch": torch.__version__,
                **self.versions,
                "device": self.device,
                "source": self.source,
                "adapted": self.adapted,
                "classes": list(self.classes),
                "num_classes": len(self.classes),
                "checkpoint_path": str(self.checkpoint),
                "checkpoint_sha256": _sha256(self.checkpoint),
            },
            "output_semantics": "Per-pixel class index in [0, num_classes - 1]. No calibrated per-pixel uncertainty is exported.",
        }

    def write_provenance(self, path: str | os.PathLike) -> Path:
        target = Path(path)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(json.dumps(self.provenance(), indent=2) + "\n")
        return target

**Module 3/3:** `src/dimer_swin_segmentation/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Deterministic in-code sample data: ADE20K demonstration scene and the custom adaptation dataset.

Nothing here is downloaded and nothing needs external dependencies beyond Pillow and numpy,
so the tutorial's default path has zero network dataset dependencies and the same generators
are exercised by the repository's unit tests.

Two separate label vocabularies live here and must not be conflated:

* ``tutorial_scene`` returns an RGB scene for demonstrating the pretrained ADE20K 150-class model.
* ``synthetic_segmentation_dataset`` returns records labelled with ``ADAPT_CLASSES``, a three-class
  vocabulary (``background``, ``road``, ``structure``) that represents a custom target deployment domain.
  A model fine-tuned on this dataset predicts only these three classes.
"""

from __future__ import annotations

import math
from collections.abc import Sequence
from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image, ImageDraw

# standalone rewrite (build_notebook.py): `from .metrics import IGNORE_INDEX` removed — names are kernel globals defined by the carried modules

ADE20K_SCENE_SIZE = (512, 384)
ADAPT_SCENE_SIZE = (512, 512)

# The custom adaptation vocabulary. Deliberately distinct from ADE20K indices.
# A model re-headed and fine-tuned on this dataset predicts these three classes.
ADAPT_CLASSES: tuple[str, ...] = ("background", "road", "structure")


def tutorial_scene(
    width: int = ADE20K_SCENE_SIZE[0], height: int = ADE20K_SCENE_SIZE[1]
) -> Image.Image:
    """The ADE20K demonstration scene: deterministic gradient background plus flat shapes.

    Matches the fixed tutorial fixture used to demonstrate pretrained inference.
    """
    ramp = np.linspace(0.0, 255.0, width)
    red = np.tile(ramp, (height, 1))
    green = np.tile(np.linspace(0.0, 255.0, height)[:, None], (1, width))
    blue = (red + green) / 2.0
    array = np.rint(np.stack([red, green, blue], axis=-1)).astype(np.uint8)
    scene = Image.fromarray(array, mode="RGB")
    draw = ImageDraw.Draw(scene)
    draw.rectangle([40, int(height * 0.625), 220, int(height * 0.9375)], fill=(20, 20, 20))
    draw.ellipse([300, int(height * 0.156), 460, int(height * 0.573)], fill=(240, 240, 240))
    draw.polygon(
        [(260, int(height * 0.963)), (330, int(height * 0.651)), (400, int(height * 0.963))],
        fill=(30, 90, 200),
    )
    return scene


def generate_scene(
    index: int,
    *,
    width: int = ADAPT_SCENE_SIZE[0],
    height: int = ADAPT_SCENE_SIZE[1],
    seed: int = 20260915,
) -> tuple[Image.Image, np.ndarray]:
    """Generate a single synthetic image and its exact pixel-aligned ground truth mask.

    Classes:
      0: background (sky / environment)
      1: road (traversable ground plane)
      2: structure (geometric buildings / block entities)
    """
    rng = np.random.default_rng(seed + index * 101)

    # Base background (class 0)
    bg_r = int(rng.integers(120, 160))
    bg_g = int(rng.integers(170, 210))
    bg_b = int(rng.integers(210, 250))
    img = Image.new("RGB", (width, height), (bg_r, bg_g, bg_b))
    d = ImageDraw.Draw(img)
    mask = np.zeros((height, width), dtype=np.uint8)

    # Road / ground plane (class 1)
    road_top = int(height * rng.uniform(0.55, 0.65))
    road_color = int(rng.integers(60, 90))
    d.rectangle([0, road_top, width, height], fill=(road_color, road_color, road_color))
    mask[road_top:height, :] = 1

    # Add 1 to 3 structures (class 2) seated on or above the ground plane
    n_structures = int(rng.integers(1, 4))
    for s_idx in range(n_structures):
        sw = max(4, int(rng.integers(max(4, int(width * 0.12)), max(5, int(width * 0.28)))))
        sh = max(4, int(rng.integers(max(4, int(height * 0.15)), max(5, int(height * 0.35)))))
        low_x = int(width * 0.05 + s_idx * width * 0.28)
        high_x = int(min(width - sw - 2, (s_idx + 1) * width * 0.32))
        if high_x <= low_x:
            sx = max(2, min(low_x, max(2, width - sw - 2)))
        else:
            sx = int(rng.integers(low_x, high_x))
        sx = max(2, min(sx, max(2, width - sw - 2)))
        sy = road_top - int(sh * rng.uniform(0.6, 0.9))
        sy = max(5, min(sy, road_top - 5))
        sy_bottom = min(height - 2, sy + sh)

        st_r = int(rng.integers(160, 220))
        st_g = int(rng.integers(30, 80))
        st_b = int(rng.integers(30, 80))
        d.rectangle([sx, sy, sx + sw, sy_bottom], fill=(st_r, st_g, st_b), outline=(255, 255, 255), width=2)
        mask[sy:sy_bottom, sx : sx + sw] = 2

        # Optional roof triangle
        if rng.random() > 0.4:
            peak_y = max(5, sy - int(sh * 0.3))
            peak_x = sx + sw // 2
            roof_pts = [(sx - 4, sy), (peak_x, peak_y), (sx + sw + 4, sy)]
            d.polygon(roof_pts, fill=(max(0, st_r - 40), max(0, st_g - 20), max(0, st_b - 20)))
            # Compute triangle mask using polygon rasterization
            roof_img = Image.new("L", (width, height), 0)
            roof_draw = ImageDraw.Draw(roof_img)
            roof_draw.polygon(roof_pts, fill=1)
            roof_arr = np.array(roof_img, dtype=bool)
            mask[roof_arr] = 2

    return img, mask


def synthetic_segmentation_dataset(
    n_samples: int = 24,
    *,
    width: int = ADAPT_SCENE_SIZE[0],
    height: int = ADAPT_SCENE_SIZE[1],
    seed: int = 20260915,
) -> list[dict[str, Any]]:
    """Produce a deterministic multi-image segmentation adaptation dataset.

    Returns a list of dicts with keys:
    - ``id``: unique string identifier
    - ``image``: PIL.Image in RGB mode
    - ``mask``: 2D uint8 numpy array with pixel values in 0..len(ADAPT_CLASSES)-1
    """
    records = []
    for i in range(n_samples):
        img, mask = generate_scene(i, width=width, height=height, seed=seed)
        records.append({
            "id": f"scene_{i:03d}",
            "image": img,
            "mask": mask,
        })
    return records


def split_dataset(
    records: Sequence[dict[str, Any]],
    *,
    val_fraction: float = 0.25,
    seed: int = 42,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    """Deterministically partition records into train and validation splits."""
    n = len(records)
    if n < 2:
        raise ValueError(f"Need at least 2 records to partition; got {n}.")
    if not (0.0 < val_fraction < 1.0):
        raise ValueError(f"val_fraction must be in (0, 1); got {val_fraction}.")

    n_val = max(1, math.floor(n * val_fraction))
    rng = np.random.default_rng(seed)
    indices = rng.permutation(n)
    val_idx = set(indices[:n_val])

    train_split = [r for i, r in enumerate(records) if i not in val_idx]
    val_split = [r for i, r in enumerate(records) if i in val_idx]
    return train_split, val_split


def validate_dataset(
    records: Sequence[dict[str, Any]],
    class_names: Sequence[str],
    *,
    epochs: int = 1,
    max_pixels: int = 64_000_000,
) -> dict[str, Any]:
    """Validate that ``records`` adhere to the ``core.dataset.vision.raster-mask`` contract.

    Raises ValueError on schema violations, shape mismatches, or invalid class indices.
    """
    if not records:
        raise ValueError("Dataset cannot be empty; at least 1 record required.")
    if not class_names:
        raise ValueError("class_names cannot be empty.")
    if epochs < 1:
        raise ValueError(f"epochs must be >= 1, got {epochs}.")

    num_classes = len(class_names)
    observed_classes: set[int] = set()
    validated_records = []

    for idx, record in enumerate(records):
        rec_id = str(record.get("id", f"sample_{idx}"))
        if "image" not in record or "mask" not in record:
            raise ValueError(f"Record {rec_id} missing 'image' or 'mask' field.")

        img_raw = record["image"]
        if isinstance(img_raw, (str, Path)):
            img_path = Path(img_raw)
            if not img_path.is_file():
                raise ValueError(f"Record {rec_id}: image file not found: {img_path}")
            with Image.open(img_path) as opened:
                img = opened.convert("RGB")
        elif isinstance(img_raw, Image.Image):
            img = img_raw.convert("RGB")
        else:
            raise TypeError(f"Record {rec_id}: unexpected image type {type(img_raw)}.")

        width, height = img.size
        if width < 1 or height < 1:
            raise ValueError(f"Record {rec_id}: dimensions must be positive; got {width}x{height}.")
        if width * height > max_pixels:
            raise ValueError(f"Record {rec_id}: {width * height} pixels exceeds ceiling {max_pixels}.")

        mask_raw = record["mask"]
        if isinstance(mask_raw, (str, Path)):
            mask_path = Path(mask_raw)
            if not mask_path.is_file():
                raise ValueError(f"Record {rec_id}: mask file not found: {mask_path}")
            with Image.open(mask_path) as opened_m:
                mask = np.array(opened_m)
        elif isinstance(mask_raw, Image.Image):
            mask = np.array(mask_raw)
        elif isinstance(mask_raw, np.ndarray):
            mask = mask_raw
        else:
            raise TypeError(f"Record {rec_id}: unexpected mask type {type(mask_raw)}.")

        if mask.ndim != 2:
            raise ValueError(f"Record {rec_id}: mask must be 2-D; got shape {mask.shape}.")
        if mask.shape != (height, width):
            raise ValueError(
                f"Record {rec_id}: mask shape {mask.shape} != image height/width ({height}, {width})."
            )

        unique_vals = np.unique(mask)
        for val in unique_vals:
            if val != IGNORE_INDEX:
                if val < 0 or val >= num_classes:
                    raise ValueError(
                        f"Record {rec_id}: mask contains class index {val} "
                        f"outside valid range 0..{num_classes - 1}."
                    )
                observed_classes.add(int(val))

        validated_records.append({
            "id": rec_id,
            "width": width,
            "height": height,
            "classes_present": [int(v) for v in unique_vals if v != IGNORE_INDEX],
        })

    return {
        "n_records": len(records),
        "class_names": list(class_names),
        "observed_classes": sorted(observed_classes),
        "schema": "core.dataset.vision.raster-mask",
        "verdict": "accepted",
    }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `1`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the OpenMMLab checkpoint host (`download.openmmlab.com`) **at MMSegmentation release-tag commit `c685fe6767c4…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `DimerSwinSegmenter.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "swin-t-upernet-ade20k",
  "modelId": "open-mmlab/mmsegmentation:swin-tiny-patch4-window7-in1k-pre_upernet_8xb2-160k_ade20k-512x512",
  "revision": "c685fe6767c4cadf6b051983ca6208f1b9d1ccb8",
  "files": [
    {
      "path": "upernet_swin_tiny_patch4_window7_512x512_160k_ade20k_pretrain_224x224_1K_20210531_112542-e380ad3e.pth",
      "bytes": 240154742,
      "sha256": "e380ad3e5d94060d89e4b62b5d393cdcc7f1f3406b1d46bcab547d3c276b6064"
    }
  ],
  "totalBytes": 240154742
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = DimerSwinSegmenter.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Confirm the qualified runtime

The carried package fails closed on version drift: `verify_runtime_versions` compares the installed `mmseg`, `mmcv` and `mmengine` distributions with the versions pinned in `MODEL_SPEC`, and this cell additionally asserts the Python 3.10 interpreter the OpenMMLab wheels were built for. Look for a dictionary reporting Python 3.10.x, `torch` 2.1.2+cpu, MMSegmentation 1.2.2, MMCV 2.1.0, MMEngine 0.10.7, 150 classes, the verified checkpoint file name and the `local-snapshot` source.

In [ ]:
import platform
import sys
import numpy
import torch

if sys.version_info[:2] != (3, 10):
    raise RuntimeError(f'Python 3.10 is required by the qualified OpenMMLab runtime (see Prerequisites); this kernel is {sys.version.split()[0]}. Use a Python 3.10 kernel.')
print({'python': platform.python_version(), 'torch': torch.__version__, 'numpy': numpy.__version__, **pipe.versions, 'classes': len(pipe.classes), 'checkpoint': pipe.checkpoint.name, 'source': pipe.source, 'device': pipe.device})

## 5. Demonstrate pretrained ADE20K capability

Before adapting to custom classes, this cell demonstrates the base model on a deterministic 512×384 demonstration scene (or an optional ADE20K fixture / uploaded image). `predict` runs the image through the pretrained 150-class UPerNet head and emits a 2-D semantic mask. Look for the top predicted classes and confirmed mask dimensions.

In [ ]:
import os
from pathlib import Path
from PIL import Image

sample_dir = Path('sample')
sample_dir.mkdir(exist_ok=True)

USE_BYOD_IMAGE = False  # @param {"type":"boolean"}
USE_ADE20K_FIXTURES = False  # @param {"type":"boolean"}

if USE_BYOD_IMAGE:
    from google.colab import files
    uploaded = files.upload()
    upload_name = next(iter(uploaded))
    demo_image_path = sample_dir / Path(upload_name).name
    demo_image_path.write_bytes(uploaded[upload_name])
    demo_img = Image.open(demo_image_path).convert('RGB')
elif USE_ADE20K_FIXTURES:
    import urllib.request
    FIXTURES_COMMIT = '850d349e5038f291284e7999fcacbedc0922534b'
    fixture_url = f'https://huggingface.co/datasets/hf-internal-testing/fixtures_ade20k/resolve/{FIXTURES_COMMIT}/ADE_val_00000001.jpg'
    demo_image_path = sample_dir / 'ADE_val_00000001.jpg'
    if not demo_image_path.exists():
        urllib.request.urlretrieve(fixture_url, demo_image_path)
    demo_img = Image.open(demo_image_path).convert('RGB')
else:
    demo_img = tutorial_scene()
    demo_image_path = sample_dir / 'ade20k_demo_scene_512x384.png'
    demo_img.save(demo_image_path)

pretrained_result = pipe.predict(demo_img)
os.makedirs('outputs', exist_ok=True)
pretrained_mask_path = pretrained_result.save_mask('outputs/swin_segmentation_pretrained_scene_semantic.png')
print({**pretrained_result.summary(), 'pretrained_classes_top': [pipe.classes[c] for c in pretrained_result.classes_present[:5]], 'saved_mask': str(pretrained_mask_path)})

## 6. Generate and validate the custom segmentation dataset

The adaptation dataset conforms to the `core.dataset.vision.raster-mask` contract: 24 synthetic multi-object scenes with exact pixel-aligned ground truth masks over `ADAPT_CLASSES = ('background', 'road', 'structure')`. `validate_dataset` asserts positive dimensions, image/mask shape registration, and class index validity. The manifest is written to `outputs/swin_segmentation_input_manifest.json` along with a rejected invalid-probe finding.

In [ ]:
import json

USE_BYOD_DATASET = False  # @param {"type":"boolean"}

if USE_BYOD_DATASET:
    print('BYOD dataset enabled.')
else:
    dataset_records = synthetic_segmentation_dataset(24, seed=DEFAULT_ADAPT_SEED)

dataset_summary = validate_dataset(dataset_records, ADAPT_CLASSES)

input_manifest = {
    'schema': dataset_summary['schema'],
    'n_records': dataset_summary['n_records'],
    'class_names': dataset_summary['class_names'],
    'observed_classes': dataset_summary['observed_classes'],
    'verdict': dataset_summary['verdict'],
    'findings': [],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
}

# Demonstrate rejection on an invalid input
try:
    validate_dataset([], ADAPT_CLASSES)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'empty-dataset-probe', 'verdict': 'rejected', 'message': str(exc)})

with open('outputs/swin_segmentation_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)

print(json.dumps(input_manifest, indent=2))

## 7. Partition the dataset into train and validation splits

`split_dataset` deterministically partitions the 24 records into 18 training examples and 6 validation examples (25% holdout) under NOTEBOOK_SPEC 2.0 DAT14. Look for disjoint subsets with verified IDs.

In [ ]:
train_records, val_records = split_dataset(dataset_records, val_fraction=0.25, seed=42)
print({
    'total_records': len(dataset_records),
    'train_records': len(train_records),
    'val_records': len(val_records),
    'classes': list(ADAPT_CLASSES),
    'train_ids': [r['id'] for r in train_records[:4]],
    'val_ids': [r['id'] for r in val_records],
})

## 8. Re-head the architecture & measure the pre-adaptation baseline

Before training on the custom 3-class vocabulary, `rehead_model` replaces `decode_head.conv_seg` (512→3) and `auxiliary_head.conv_seg` (256→3), and `freeze_backbone` sets `requires_grad = False` on the 28.3M Swin-T backbone. We evaluate the newly re-headed model on the held-out validation split to record the pre-adaptation baseline.

In [ ]:
rehead_model(pipe.model, ADAPT_CLASSES, seed=DEFAULT_ADAPT_SEED)
pipe.classes = tuple(ADAPT_CLASSES)
frozen_params = freeze_backbone(pipe.model)

pre_adapt_report = pipe.evaluate(val_records, sample_kind='synthetic-val')
with open('outputs/swin_segmentation_pre_adapt_evaluation.json', 'w', encoding='utf-8') as handle:
    json.dump(pre_adapt_report, handle, indent=2, ensure_ascii=False)

print({
    'stage': 'pre-adaptation baseline',
    'frozen_backbone_params': frozen_params,
    'classes': list(pipe.classes),
    'miou': next((m['value'] for m in pre_adapt_report['metrics'] if m['metric'] == 'miou'), None),
    'pixel_accuracy': next((m['value'] for m in pre_adapt_report['metrics'] if m['metric'] == 'pixel_accuracy'), None),
    'verdict': pre_adapt_report['verdict'],
})

## 9. Run bounded in-process fine-tuning

`finetune` executes an in-process AdamW optimization loop over the re-headed decode and auxiliary heads using native MMSegmentation cross-entropy loss and data preprocessing. With the backbone frozen, the loop executes 3 epochs over 18 training examples (batch size 4), logging monotonic loss descent.

In [ ]:
finetune_summary = pipe.finetune(
    train_records,
    epochs=DEFAULT_ADAPT_EPOCHS,
    batch_size=DEFAULT_ADAPT_BATCH_SIZE,
    learning_rate=DEFAULT_ADAPT_LEARNING_RATE,
    seed=DEFAULT_ADAPT_SEED,
    freeze_backbone_weights=True,
    progress=lambda p: print(f"Epoch {p['epoch']}/{p['epochs']} — loss: {p['loss']:.4f}"),
)
print(json.dumps({k: v for k, v in finetune_summary.items() if k != 'trainable_parameters'}, indent=2))

## 10. Evaluate adapted model on the held-out split

`pipe.evaluate` scores the adapted model on the held-out validation split. `semantic_iou` calculates aggregate mIoU, pixel accuracy, and per-class IoU against the `majority_class_baseline`. Look for substantial mIoU gain over the pre-adaptation baseline and clear class separation.

In [ ]:
post_adapt_report = pipe.evaluate(val_records, sample_kind='synthetic-val')
with open('outputs/swin_segmentation_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(post_adapt_report, handle, indent=2, ensure_ascii=False)

pre_miou = next((m['value'] for m in pre_adapt_report['metrics'] if m['metric'] == 'miou'), 0.0)
post_miou = next((m['value'] for m in post_adapt_report['metrics'] if m['metric'] == 'miou'), 0.0)
pre_acc = next((m['value'] for m in pre_adapt_report['metrics'] if m['metric'] == 'pixel_accuracy'), 0.0)
post_acc = next((m['value'] for m in post_adapt_report['metrics'] if m['metric'] == 'pixel_accuracy'), 0.0)

print(json.dumps({
    'miou_before': pre_miou,
    'miou_after': post_miou,
    'pixel_acc_before': pre_acc,
    'pixel_acc_after': post_acc,
    'majority_baseline_miou': post_adapt_report['baselines'][0]['miou'],
    'majority_baseline_acc': post_adapt_report['baselines'][0]['pixel_accuracy'],
    'classes_with_union': post_adapt_report['classes_with_union'],
    'verdict': post_adapt_report['verdict'],
}, indent=2))
for row in post_adapt_report['per_class']:
    print(f"class {row['class_id']:>2} {pipe.classes[row['class_id']]:<12} IoU {row['iou']:.4f}  inter {row['intersection_pixels']} px  union {row['union_pixels']} px")

## 11. Run inference on an unseen test scene

This cell generates a fresh unseen test scene (`generate_scene(99)`), runs inference with the adapted model, scores IoU against the known test mask, and exports `outputs/swin_segmentation_test_scene_adapted_semantic.png`.

In [ ]:
test_img, test_mask = generate_scene(99, seed=DEFAULT_ADAPT_SEED)
test_image_path = sample_dir / 'test_scene_unseen_512x512.png'
test_img.save(test_image_path)

adapted_test_result = pipe.predict(test_img)
adapted_mask_path = adapted_test_result.save_mask('outputs/swin_segmentation_test_scene_adapted_semantic.png')

test_scored = semantic_iou([adapted_test_result.mask], [test_mask], num_classes=len(pipe.classes))
print({
    **adapted_test_result.summary(),
    'mask_file': str(adapted_mask_path),
    'test_scene_miou': test_scored['miou'],
    'test_scene_pixel_acc': test_scored['pixel_accuracy'],
    'classes_predicted': [pipe.classes[c] for c in adapted_test_result.classes_present],
})

## 12. Export portable adapted artifact

`pipe.save_artifact` exports the adapted weights, class vocabulary, base model identity, and provenance as a standalone `.pt` artifact (`outputs/swin-segmentation-adapter-v1.pt`) under NOTEBOOK_SPEC 2.0 ART1–ART8.

In [ ]:
adapter_path = Path('outputs/swin-segmentation-adapter-v1.pt')
artifact_descriptor = pipe.save_artifact(adapter_path, notes='Swin-T UPerNet 3-class adapted segmentation model')
print(json.dumps(artifact_descriptor, indent=2))

## 13. Fresh reload & numerical verification

Under NOTEBOOK_SPEC 2.0 VER1–VER5, `DimerSwinSegmenter.load_artifact` reloads the exported weights using `weights_only=True`, constructs a fresh segmenter instance, and asserts that predicted masks on the test scene are numerically identical (`numpy.testing.assert_array_equal`).

In [ ]:
reloaded_pipe = DimerSwinSegmenter.load_artifact(adapter_path, weights_dir=WEIGHTS_DIR)
reloaded_result = reloaded_pipe.predict(test_img)

numpy.testing.assert_array_equal(
    reloaded_result.mask,
    adapted_test_result.mask,
    err_msg='Reloaded model mask must be numerically identical to adapted model mask',
)
print({
    'reloaded_source': reloaded_pipe.source,
    'reloaded_classes': list(reloaded_pipe.classes),
    'adapted_status': reloaded_pipe.adapted,
    'exact_mask_match': True,
    'shape': list(reloaded_result.mask.shape),
})

## 14. Export outputs and provenance

Writes machine-readable outputs: class coverage CSV (`outputs/swin_segmentation_class_coverage.csv`), summary result JSON (`outputs/swin_segmentation_result.json`), and comprehensive provenance (`outputs/swin_segmentation_provenance.json`).

In [ ]:
import csv

val_predictions = [pipe.predict(r['image']) for r in val_records]
coverage = []
for r, pred in zip(val_records, val_predictions, strict=True):
    values, counts = numpy.unique(pred.mask, return_counts=True)
    order = numpy.argsort(counts)[::-1]
    for class_id, pixels in zip(values[order].tolist(), counts[order].tolist(), strict=True):
        coverage.append({
            'image_id': r['id'],
            'class_id': int(class_id),
            'class_name': pipe.classes[int(class_id)],
            'pixels': int(pixels),
            'fraction': float(pixels / pred.mask.size),
        })

with open('outputs/swin_segmentation_class_coverage.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['image_id', 'class_id', 'class_name', 'pixels', 'fraction'])
    for row in coverage:
        writer.writerow([row['image_id'], row['class_id'], row['class_name'], row['pixels'], f"{row['fraction']:.6f}"])

result_payload = {
    'task': 'semantic-segmentation',
    'profile': 'E2E',
    'base_model_id': MODEL_ID,
    'base_model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'checkpoint_sha256': MODEL_SPEC['checkpoint_sha256'],
    'adapted_vocabulary': list(pipe.classes),
    'num_classes': len(pipe.classes),
    'pre_adaptation_miou': pre_miou,
    'post_adaptation_miou': post_miou,
    'post_adaptation_pixel_acc': post_acc,
    'majority_baseline_miou': post_adapt_report['baselines'][0]['miou'],
    'test_scene_miou': test_scored['miou'],
    'adapter_artifact': artifact_descriptor,
    'reloaded_verification': {'exact_mask_match': True},
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'numpy': numpy.__version__,
        **pipe.versions,
        'device': pipe.device,
    },
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'notebook_source': NOTEBOOK_SOURCE,
}

with open('outputs/swin_segmentation_result.json', 'w', encoding='utf-8') as handle:
    json.dump(result_payload, handle, indent=2, ensure_ascii=False)

provenance_path = pipe.write_provenance('outputs/swin_segmentation_provenance.json')
print(sorted(os.listdir('outputs')))

## Interpretation and limits

Each pixel receives one class index from the target vocabulary (`background`, `road`, `structure`) by per-pixel argmax; the API exposes **no per-pixel confidence** and the package ships no threshold. The synthetic dataset demonstrates that the architecture can be re-headed and adapted to custom segmentation classes in-process with a frozen backbone, but these numbers do not represent real-world photographic scene segmentation.

Successful execution proves that the recorded repository revision's package, carried in this notebook, can acquire and digest-verify the pinned OpenMMLab checkpoint, assert the qualified Python 3.10 runtime, validate the dataset contract, re-head the decode heads, run a bounded fine-tune, evaluate mIoU against a majority baseline, export the adapted artifact, and reload it with exact numerical reproducibility — without the repository being reachable. It does **not** establish benchmark superiority against other semantic-segmentation architectures or full ADE20K benchmarks.

**Next experiments:** test fine-tuning with different learning rates (`5e-5`, `2e-4`); enable `USE_BYOD_DATASET` with your own domain-specific images and segmentation masks; experiment with unfreezing the later stages of the Swin-T backbone.

## References

- Repository README: https://github.com/kurtvalcorza/swin-segmentation-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/swin-segmentation-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/swin-segmentation-pipeline/blob/main/docs/WEIGHTS.md
- Pinned checkpoint (OpenMMLab host): https://download.openmmlab.com/mmsegmentation/v0.5/swin/upernet_swin_tiny_patch4_window7_512x512_160k_ade20k_pretrain_224x224_1K/upernet_swin_tiny_patch4_window7_512x512_160k_ade20k_pretrain_224x224_1K_20210531_112542-e380ad3e.pth
- Config source (MMSegmentation v1.2.2, `configs/swin`): https://github.com/open-mmlab/mmsegmentation/tree/v1.2.2/configs/swin
- Upstream project: https://github.com/microsoft/Swin-Transformer
- Swin Transformer: Hierarchical Vision Transformer using Shifted Windows: https://arxiv.org/abs/2103.14030
- Unified Perceptual Parsing for Scene Understanding (UPerNet): https://arxiv.org/abs/1807.10221